# 06. Inference Demo

В этом ноутбуке показываем, как использовать обученную модель для предсказания кредитного риска на новых клиентах.

Основная задача:

1. Загрузить финальную модель и артефакты.
2. Подготовить входные признаки клиента.
3. Получить вероятность дефолта.
4. Применить risk policy.
5. Сформировать inference response, который можно использовать в API.

In [1]:
import json
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)

In [2]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("MODELS_DIR:", MODELS_DIR)

PROJECT_ROOT: /Users/artem/PycharmProjects/credit-risk-scoring-service
PROCESSED_DIR: /Users/artem/PycharmProjects/credit-risk-scoring-service/data/processed
MODELS_DIR: /Users/artem/PycharmProjects/credit-risk-scoring-service/models


## 1. Загрузка модели и артефактов

Используем финальную модель, выбранную в `05_model_evaluation.ipynb`.

По результатам evaluation основной моделью выбран `LightGBM`.

In [3]:
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


final_model_config_path = MODELS_DIR / "final_model_config.json"
lightgbm_model_path = MODELS_DIR / "lightgbm_model.pkl"
lightgbm_feature_list_path = MODELS_DIR / "lightgbm_feature_list.json"
lightgbm_params_path = MODELS_DIR / "lightgbm_params.json"
lightgbm_metrics_path = MODELS_DIR / "lightgbm_metrics.json"

final_model_config = load_json(final_model_config_path)
feature_cols = load_json(lightgbm_feature_list_path)
lightgbm_params = load_json(lightgbm_params_path)
lightgbm_metrics = load_json(lightgbm_metrics_path)

with open(lightgbm_model_path, "rb") as f:
    model = pickle.load(f)

print("Selected model:", final_model_config["selected_model"])
print("Number of features:", len(feature_cols))
print("Threshold:", final_model_config["threshold"])
print("Valid ROC-AUC:", lightgbm_metrics["valid_roc_auc"])
print("Valid PR-AUC:", lightgbm_metrics["valid_pr_auc"])

Selected model: LightGBM
Number of features: 322
Threshold: 0.6500000000000001
Valid ROC-AUC: 0.7883975154960883
Valid PR-AUC: 0.2866316232885762


In [4]:
test_path = PROCESSED_DIR / "test_features.parquet"

test = pd.read_parquet(test_path)

ID_COL = "SK_ID_CURR"

print("test:", test.shape)
test.head()

test: (48744, 323)


,SK_ID_CURR,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,REGION_RATING_CLIENT,REGION_RATING_CLIENT_W_CITY,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,REG_REGION_NOT_LIVE_REGION,REG_REGION_NOT_WORK_REGION,LIVE_REGION_NOT_WORK_REGION,REG_CITY_NOT_LIVE_CITY,REG_CITY_NOT_WORK_CITY,LIVE_CITY_NOT_WORK_CITY,ORGANIZATION_TYPE,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,APARTMENTS_AVG,BASEMENTAREA_AVG,YEARS_BEGINEXPLUATATION_AVG,YEARS_BUILD_AVG,COMMONAREA_AVG,ELEVATORS_AVG,ENTRANCES_AVG,FLOORSMAX_AVG,FLOORSMIN_AVG,LANDAREA_AVG,LIVINGAPARTMENTS_AVG,LIVINGAREA_AVG,NONLIVINGAPARTMENTS_AVG,NONLIVINGAREA_AVG,APARTMENTS_MODE,BASEMENTAREA_MODE,YEARS_BEGINEXPLUATATION_MODE,YEARS_BUILD_MODE,COMMONAREA_MODE,ELEVATORS_MODE,ENTRANCES_MODE,FLOORSMAX_MODE,FLOORSMIN_MODE,LANDAREA_MODE,LIVINGAPARTMENTS_MODE,LIVINGAREA_MODE,NONLIVINGAPARTMENTS_MODE,NONLIVINGAREA_MODE,APARTMENTS_MEDI,BASEMENTAREA_MEDI,YEARS_BEGINEXPLUATATION_MEDI,YEARS_BUILD_MEDI,COMMONAREA_MEDI,ELEVATORS_MEDI,ENTRANCES_MEDI,FLOORSMAX_MEDI,FLOORSMIN_MEDI,LANDAREA_MEDI,LIVINGAPARTMENTS_MEDI,LIVINGAREA_MEDI,NONLIVINGAPARTMENTS_MEDI,NONLIVINGAREA_MEDI,FONDKAPREMONT_MODE,HOUSETYPE_MODE,TOTALAREA_MODE,WALLSMATERIAL_MODE,EMERGENCYSTATE_MODE,OBS_30_CNT_SOCIAL_CIRCLE,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,...,PREV_NAME_CONTRACT_TYPE_Cash loans,PREV_NAME_CONTRACT_TYPE_Consumer loans,PREV_NAME_CONTRACT_TYPE_Revolving loans,PREV_NAME_CONTRACT_TYPE_XNA,PREV_NAME_CONTRACT_TYPE_nan,PREV_NAME_CLIENT_TYPE_New,PREV_NAME_CLIENT_TYPE_Refreshed,PREV_NAME_CLIENT_TYPE_Repeater,PREV_NAME_CLIENT_TYPE_XNA,PREV_NAME_CLIENT_TYPE_nan,PREV_CHANNEL_TYPE_AP+ (Cash loan),PREV_CHANNEL_TYPE_Car dealer,PREV_CHANNEL_TYPE_Channel of corporate sales,PREV_CHANNEL_TYPE_Contact center,PREV_CHANNEL_TYPE_Country-wide,PREV_CHANNEL_TYPE_Credit and cash offices,PREV_CHANNEL_TYPE_Regional / Local,PREV_CHANNEL_TYPE_Stone,PREV_CHANNEL_TYPE_nan,PREV_NAME_YIELD_GROUP_XNA,PREV_NAME_YIELD_GROUP_high,PREV_NAME_YIELD_GROUP_low_action,PREV_NAME_YIELD_GROUP_low_normal,PREV_NAME_YIELD_GROUP_middle,PREV_NAME_YIELD_GROUP_nan,PREV_APPROVED_RATIO,PREV_REFUSED_RATIO,PREV_CANCELED_RATIO,POS_RECORDS_COUNT,POS_PREV_UNIQUE_COUNT,POS_MONTHS_BALANCE_MEAN,POS_MONTHS_BALANCE_MIN,POS_MONTHS_BALANCE_MAX,POS_CNT_INSTALMENT_MEAN,POS_CNT_INSTALMENT_MAX,POS_CNT_INSTALMENT_FUTURE_MEAN,POS_CNT_INSTALMENT_FUTURE_MAX,POS_SK_DPD_MEAN,POS_SK_DPD_MAX,POS_SK_DPD_DEF_MEAN,POS_SK_DPD_DEF_MAX,POS_NAME_CONTRACT_STATUS_Active,POS_NAME_CONTRACT_STATUS_Amortized debt,POS_NAME_CONTRACT_STATUS_Approved,POS_NAME_CONTRACT_STATUS_Canceled,POS_NAME_CONTRACT_STATUS_Completed,POS_NAME_CONTRACT_STATUS_Demand,POS_NAME_CONTRACT_STATUS_Returned to the store,POS_NAME_CONTRACT_STATUS_Signed,POS_NAME_CONTRACT_STATUS_XNA,POS_NAME_CONTRACT_STATUS_nan,INSTAL_RECORDS_COUNT,INSTAL_PREV_UNIQUE_COUNT,INSTAL_NUM_INSTALMENT_VERSION_MEAN,INSTAL_NUM_INSTALMENT_NUMBER_MAX,INSTAL_DAYS_INSTALMENT_MEAN,INSTAL_DAYS_ENTRY_PAYMENT_MEAN,INSTAL_PAYMENT_DELAY_MEAN,INSTAL_PAYMENT_DELAY_MAX,INSTAL_PAYMENT_DELAY_SUM,INSTAL_PAYMENT_DIFF_MEAN,INSTAL_PAYMENT_DIFF_SUM,INSTAL_LATE_PAYMENT_MEAN,INSTAL_LATE_PAYMENT_SUM,INSTAL_UNDERPAYMENT_MEAN,INSTAL_UNDERPAYMENT_SUM,INSTAL_AMT_INSTALMENT_MEAN,INSTAL_AMT_INSTALMENT_SUM,INSTAL_AMT_PAYMENT_MEAN,INSTAL_AMT_PAYMENT_SUM,CC_RECORDS_COUNT,CC_PREV_UNIQUE_COUNT,CC_MONTHS_BALANCE_MEAN,CC_MONTHS_BALANCE_MIN,CC_MONTHS_BALANCE_MAX,CC_AMT_BALANCE_MEAN,CC_AMT_BALANCE_MAX,CC_AMT_CREDIT_LIMIT_ACTUAL_MEAN,CC_AMT_CREDIT_LIMIT_ACTUAL_MAX,CC_AMT_DRAWINGS_CURRENT_MEAN,CC_AMT_DR

In [5]:
test_ids = test[ID_COL].copy()
X_test = test[feature_cols].copy()

X_test = X_test.replace([np.inf, -np.inf], np.nan)

categorical_features = X_test.select_dtypes(
    exclude=["number", "bool"]
).columns.tolist()

for col in categorical_features:
    X_test[col] = X_test[col].astype("category")

print("X_test:", X_test.shape)
print("Categorical features:", len(categorical_features))

X_test: (48744, 322)
Categorical features: 16


## 2. Inference для одного клиента

Сначала покажем inference на одном клиенте из test-выборки.

Это имитирует ситуацию, когда API получает признаки одного клиента и должен вернуть:

- вероятность дефолта;
- risk decision;
- risk grade;
- основные служебные поля.

In [6]:
client_idx = 0

client_id = int(test_ids.iloc[client_idx])
client_features = X_test.iloc[[client_idx]].copy()

print("Client ID:", client_id)
print("Client features shape:", client_features.shape)

client_features.head()

Client ID: 100001
Client features shape: (1, 322)


,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,REGION_RATING_CLIENT,REGION_RATING_CLIENT_W_CITY,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,REG_REGION_NOT_LIVE_REGION,REG_REGION_NOT_WORK_REGION,LIVE_REGION_NOT_WORK_REGION,REG_CITY_NOT_LIVE_CITY,REG_CITY_NOT_WORK_CITY,LIVE_CITY_NOT_WORK_CITY,ORGANIZATION_TYPE,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,APARTMENTS_AVG,BASEMENTAREA_AVG,YEARS_BEGINEXPLUATATION_AVG,YEARS_BUILD_AVG,COMMONAREA_AVG,ELEVATORS_AVG,ENTRANCES_AVG,FLOORSMAX_AVG,FLOORSMIN_AVG,LANDAREA_AVG,LIVINGAPARTMENTS_AVG,LIVINGAREA_AVG,NONLIVINGAPARTMENTS_AVG,NONLIVINGAREA_AVG,APARTMENTS_MODE,BASEMENTAREA_MODE,YEARS_BEGINEXPLUATATION_MODE,YEARS_BUILD_MODE,COMMONAREA_MODE,ELEVATORS_MODE,ENTRANCES_MODE,FLOORSMAX_MODE,FLOORSMIN_MODE,LANDAREA_MODE,LIVINGAPARTMENTS_MODE,LIVINGAREA_MODE,NONLIVINGAPARTMENTS_MODE,NONLIVINGAREA_MODE,APARTMENTS_MEDI,BASEMENTAREA_MEDI,YEARS_BEGINEXPLUATATION_MEDI,YEARS_BUILD_MEDI,COMMONAREA_MEDI,ELEVATORS_MEDI,ENTRANCES_MEDI,FLOORSMAX_MEDI,FLOORSMIN_MEDI,LANDAREA_MEDI,LIVINGAPARTMENTS_MEDI,LIVINGAREA_MEDI,NONLIVINGAPARTMENTS_MEDI,NONLIVINGAREA_MEDI,FONDKAPREMONT_MODE,HOUSETYPE_MODE,TOTALAREA_MODE,WALLSMATERIAL_MODE,EMERGENCYSTATE_MODE,OBS_30_CNT_SOCIAL_CIRCLE,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,...,PREV_NAME_CONTRACT_TYPE_Cash loans,PREV_NAME_CONTRACT_TYPE_Consumer loans,PREV_NAME_CONTRACT_TYPE_Revolving loans,PREV_NAME_CONTRACT_TYPE_XNA,PREV_NAME_CONTRACT_TYPE_nan,PREV_NAME_CLIENT_TYPE_New,PREV_NAME_CLIENT_TYPE_Refreshed,PREV_NAME_CLIENT_TYPE_Repeater,PREV_NAME_CLIENT_TYPE_XNA,PREV_NAME_CLIENT_TYPE_nan,PREV_CHANNEL_TYPE_AP+ (Cash loan),PREV_CHANNEL_TYPE_Car dealer,PREV_CHANNEL_TYPE_Channel of corporate sales,PREV_CHANNEL_TYPE_Contact center,PREV_CHANNEL_TYPE_Country-wide,PREV_CHANNEL_TYPE_Credit and cash offices,PREV_CHANNEL_TYPE_Regional / Local,PREV_CHANNEL_TYPE_Stone,PREV_CHANNEL_TYPE_nan,PREV_NAME_YIELD_GROUP_XNA,PREV_NAME_YIELD_GROUP_high,PREV_NAME_YIELD_GROUP_low_action,PREV_NAME_YIELD_GROUP_low_normal,PREV_NAME_YIELD_GROUP_middle,PREV_NAME_YIELD_GROUP_nan,PREV_APPROVED_RATIO,PREV_REFUSED_RATIO,PREV_CANCELED_RATIO,POS_RECORDS_COUNT,POS_PREV_UNIQUE_COUNT,POS_MONTHS_BALANCE_MEAN,POS_MONTHS_BALANCE_MIN,POS_MONTHS_BALANCE_MAX,POS_CNT_INSTALMENT_MEAN,POS_CNT_INSTALMENT_MAX,POS_CNT_INSTALMENT_FUTURE_MEAN,POS_CNT_INSTALMENT_FUTURE_MAX,POS_SK_DPD_MEAN,POS_SK_DPD_MAX,POS_SK_DPD_DEF_MEAN,POS_SK_DPD_DEF_MAX,POS_NAME_CONTRACT_STATUS_Active,POS_NAME_CONTRACT_STATUS_Amortized debt,POS_NAME_CONTRACT_STATUS_Approved,POS_NAME_CONTRACT_STATUS_Canceled,POS_NAME_CONTRACT_STATUS_Completed,POS_NAME_CONTRACT_STATUS_Demand,POS_NAME_CONTRACT_STATUS_Returned to the store,POS_NAME_CONTRACT_STATUS_Signed,POS_NAME_CONTRACT_STATUS_XNA,POS_NAME_CONTRACT_STATUS_nan,INSTAL_RECORDS_COUNT,INSTAL_PREV_UNIQUE_COUNT,INSTAL_NUM_INSTALMENT_VERSION_MEAN,INSTAL_NUM_INSTALMENT_NUMBER_MAX,INSTAL_DAYS_INSTALMENT_MEAN,INSTAL_DAYS_ENTRY_PAYMENT_MEAN,INSTAL_PAYMENT_DELAY_MEAN,INSTAL_PAYMENT_DELAY_MAX,INSTAL_PAYMENT_DELAY_SUM,INSTAL_PAYMENT_DIFF_MEAN,INSTAL_PAYMENT_DIFF_SUM,INSTAL_LATE_PAYMENT_MEAN,INSTAL_LATE_PAYMENT_SUM,INSTAL_UNDERPAYMENT_MEAN,INSTAL_UNDERPAYMENT_SUM,INSTAL_AMT_INSTALMENT_MEAN,INSTAL_AMT_INSTALMENT_SUM,INSTAL_AMT_PAYMENT_MEAN,INSTAL_AMT_PAYMENT_SUM,CC_RECORDS_COUNT,CC_PREV_UNIQUE_COUNT,CC_MONTHS_BALANCE_MEAN,CC_MONTHS_BALANCE_MIN,CC_MONTHS_BALANCE_MAX,CC_AMT_BALANCE_MEAN,CC_AMT_BALANCE_MAX,CC_AMT_CREDIT_LIMIT_ACTUAL_MEAN,CC_AMT_CREDIT_LIMIT_ACTUAL_MAX,CC_AMT_DRAWINGS_CURRENT_MEAN,CC_A

In [7]:
client_pd = float(
    model.predict(
        client_features,
        num_iteration=model.best_iteration,
        validate_features=False
    )[0]
)

client_pd

0.22773147061797416

In [8]:
threshold = float(final_model_config["threshold"])

client_is_bad = client_pd >= threshold

decision = "REJECT" if client_is_bad else "APPROVE"

print("PD:", client_pd)
print("Threshold:", threshold)
print("Decision:", decision)

PD: 0.22773147061797416
Threshold: 0.6500000000000001
Decision: APPROVE


## 3. Risk policy

Risk policy переводит вероятность дефолта в понятное бизнес-решение.

В demo используем простую policy:

- `LOW` risk — низкая вероятность дефолта;
- `MEDIUM` risk — средний риск;
- `HIGH` risk — высокий риск;
- `VERY_HIGH` risk — очень высокий риск.

В реальном банке эти границы выбираются по бизнес-стоимости ошибок, approval rate и risk appetite.

In [9]:
def assign_risk_grade(pd_value):
    if pd_value < 0.20:
        return "LOW"
    elif pd_value < 0.40:
        return "MEDIUM"
    elif pd_value < 0.65:
        return "HIGH"
    else:
        return "VERY_HIGH"


def make_credit_decision(pd_value, threshold):
    if pd_value >= threshold:
        return "REJECT"
    return "APPROVE"

In [10]:
client_risk_grade = assign_risk_grade(client_pd)
client_decision = make_credit_decision(client_pd, threshold)

client_response = {
    "client_id": client_id,
    "model": final_model_config["selected_model"],
    "probability_of_default": client_pd,
    "threshold": threshold,
    "risk_grade": client_risk_grade,
    "decision": client_decision,
}

client_response

{'client_id': 100001,
 'model': 'LightGBM',
 'probability_of_default': 0.22773147061797416,
 'threshold': 0.6500000000000001,
 'risk_grade': 'MEDIUM',
 'decision': 'APPROVE'}

## 4. Batch inference

Теперь применим inference pipeline к нескольким клиентам из test-выборки.

Это показывает, как модель будет работать в batch-режиме или внутри API, если нужно обработать несколько заявок.

In [11]:
def predict_batch(model, X, ids, threshold):
    predicted_pd = model.predict(
        X,
        num_iteration=model.best_iteration,
        validate_features=False
    )

    result = pd.DataFrame({
        "client_id": ids.values,
        "probability_of_default": predicted_pd,
    })

    result["threshold"] = threshold
    result["risk_grade"] = result["probability_of_default"].apply(assign_risk_grade)
    result["decision"] = result["probability_of_default"].apply(
        lambda x: make_credit_decision(x, threshold)
    )

    return result

In [12]:
sample_size = 10

sample_features = X_test.iloc[:sample_size].copy()
sample_ids = test_ids.iloc[:sample_size].copy()

sample_predictions = predict_batch(
    model=model,
    X=sample_features,
    ids=sample_ids,
    threshold=threshold,
)

sample_predictions

,client_id,probability_of_default,threshold,risk_grade,decision
0,100001,0.227731,0.65,MEDIUM,APPROVE
1,100005,0.751526,0.65,VERY_HIGH,REJECT
2,100013,0.194432,0.65,LOW,APPROVE
3,100028,0.332313,0.65,MEDIUM,APPROVE
4,100038,0.629138,0.65,HIGH,APPROVE
5,100042,0.216774,0.65,MEDIUM,APPROVE
6,100057,0.035201,0.65,LOW,APPROVE
7,100065,0.220645,0.65,MEDIUM,APPROVE
8,100066,0.105911,0.65,LOW,APPROVE
9,100067,0.437773,0.65,HIGH,APPROVE


In [13]:
sample_predictions["decision"].value_counts()

decision
APPROVE    9
REJECT     1
Name: count, dtype: int64

In [14]:
inference_sample_path = PROCESSED_DIR / "inference_sample.json"

inference_sample = {
    "model": final_model_config["selected_model"],
    "threshold": threshold,
    "sample_predictions": sample_predictions.to_dict(orient="records"),
}

with open(inference_sample_path, "w", encoding="utf-8") as f:
    json.dump(inference_sample, f, ensure_ascii=False, indent=2)

print("Saved:", inference_sample_path)

Saved: /Users/artem/PycharmProjects/credit-risk-scoring-service/data/processed/inference_sample.json


## 5. Full test inference

Теперь применим модель ко всей test-выборке.

Сохраним файл с:

- `client_id`;
- probability of default;
- risk grade;
- decision.

Этот файл можно использовать как пример batch scoring output.

In [15]:
full_test_predictions = predict_batch(
    model=model,
    X=X_test,
    ids=test_ids,
    threshold=threshold,
)

print("full_test_predictions:", full_test_predictions.shape)

full_test_predictions.head()

full_test_predictions: (48744, 5)


,client_id,probability_of_default,threshold,risk_grade,decision
0,100001,0.227731,0.65,MEDIUM,APPROVE
1,100005,0.751526,0.65,VERY_HIGH,REJECT
2,100013,0.194432,0.65,LOW,APPROVE
3,100028,0.332313,0.65,MEDIUM,APPROVE
4,100038,0.629138,0.65,HIGH,APPROVE


In [16]:
print("Decision distribution:")
display(full_test_predictions["decision"].value_counts(normalize=True).to_frame("share"))

print("Risk grade distribution:")
display(full_test_predictions["risk_grade"].value_counts(normalize=True).to_frame("share"))

Decision distribution:


,share
decision,
APPROVE,0.880703
REJECT,0.119297


Risk grade distribution:


,share
risk_grade,
LOW,0.365522
MEDIUM,0.281963
HIGH,0.233218
VERY_HIGH,0.119297


In [17]:
full_test_predictions["probability_of_default"].describe()

count    48744.000000
mean         0.333188
std          0.224605
min          0.003112
25%          0.141531
50%          0.285076
75%          0.498033
max          0.953450
Name: probability_of_default, dtype: float64

In [18]:
full_inference_path = MODELS_DIR / "lightgbm_inference_predictions.csv"

full_test_predictions.to_csv(full_inference_path, index=False)

print("Saved:", full_inference_path)

Saved: /Users/artem/PycharmProjects/credit-risk-scoring-service/models/lightgbm_inference_predictions.csv


## 6. Example API payload

Сформируем пример входного payload и выходного response для будущего API.

В реальном API клиент отправляет признаки, а сервис возвращает вероятность дефолта, risk grade и решение.

In [19]:
example_client_raw = test.iloc[client_idx].copy()

example_request = {
    "client_id": int(example_client_raw[ID_COL]),
    "features": {
        col: (
            None if pd.isna(example_client_raw[col])
            else example_client_raw[col].item() if hasattr(example_client_raw[col], "item")
            else example_client_raw[col]
        )
        for col in feature_cols
    }
}

list(example_request["features"].items())[:10]

[('NAME_CONTRACT_TYPE', 'Cash loans'),
 ('CODE_GENDER', 'F'),
 ('FLAG_OWN_CAR', 'N'),
 ('FLAG_OWN_REALTY', 'Y'),
 ('CNT_CHILDREN', 0),
 ('AMT_INCOME_TOTAL', 135000.0),
 ('AMT_CREDIT', 568800.0),
 ('AMT_ANNUITY', 20560.5),
 ('AMT_GOODS_PRICE', 450000.0),
 ('NAME_TYPE_SUITE', 'Unaccompanied')]

In [20]:
example_response = client_response.copy()

example_payload = {
    "request": example_request,
    "response": example_response,
}

example_payload_path = PROCESSED_DIR / "inference_api_example.json"

with open(example_payload_path, "w", encoding="utf-8") as f:
    json.dump(example_payload, f, ensure_ascii=False, indent=2)

print("Saved:", example_payload_path)

Saved: /Users/artem/PycharmProjects/credit-risk-scoring-service/data/processed/inference_api_example.json


## 7. Выводы

В этом ноутбуке был собран inference demo pipeline для финальной LightGBM-модели.

Что было сделано:

1. Загружены финальная модель, список признаков, threshold и метрики.
2. Подготовлены признаки test-выборки в том же формате, что при обучении.
3. Получен prediction для одного клиента.
4. Вероятность дефолта была переведена в risk grade и business decision.
5. Выполнен batch inference для нескольких клиентов.
6. Сохранён `inference_sample.json`.
7. Выполнен полный batch scoring для test-выборки.
8. Сохранён пример API request/response.

Этот ноутбук показывает, как обученную модель можно использовать в production-like inference pipeline и далее подключить к FastAPI-сервису.

In [21]:
saved_files = [
    PROCESSED_DIR / "inference_sample.json",
    PROCESSED_DIR / "inference_api_example.json",
    MODELS_DIR / "lightgbm_inference_predictions.csv",
]

for path in saved_files:
    print(path, "->", path.exists())

/Users/artem/PycharmProjects/credit-risk-scoring-service/data/processed/inference_sample.json -> True
/Users/artem/PycharmProjects/credit-risk-scoring-service/data/processed/inference_api_example.json -> True
/Users/artem/PycharmProjects/credit-risk-scoring-service/models/lightgbm_inference_predictions.csv -> True
